# GNN Lipid Membrane Property Training (Colab Optimized)
This notebook is designed to train the Heterogeneous GNN for membrane properties using CUDA-accelerated resources in Google Colab.

### Setup Instructions:
1. Upload the `colab_lipid_gnn_subset.zip` file (created by `prepare_colab_subset.py`) to the Colab session.
2. Upload the `lipid_gnn` package folder (or clone the repository).
3. Run the cells below.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!cp "/content/drive/MyDrive/data/colab_lipid_gnn_subset.zip" .

In [ ]:
%%capture
# @title Install Dependencies
!pip install torch-geometric mdtraj mdanalysis tqdm scikit-learn matplotlib nglview

In [ ]:
import torch, torch_geometric
print(f"PyTorch: {torch.__version__}")
print(f"CUDA:    {torch.version.cuda}")
print(f"PYG:     {torch_geometric.__version__}")

In [ ]:
%%capture
# @title Complete PyG Extension Stack (Scatter, Sparse, Cluster)
import torch
import os

# Detect current versions
TORCH = torch.__version__.split('+')[0]
CUDA = torch.version.cuda.replace('.', '')

# The -f flag (find-links) points to PyG's pre-compiled binaries
!pip install torch_scatter torch_sparse torch_cluster -f https://data.pyg.org/whl/torch-2.10.0+cu128.html
#!pip install torch-scatter torch-sparse torch-cluster torch-spline-conv -f https://data.pyg.org/whl/torch-{TORCH}+{CUDA}.html

print(f"Installing for PyTorch {TORCH} and CUDA {CUDA}...")
print("PyG extensions installed successfully!")


In [ ]:
%%capture
!pip install wandb -qU

In [ ]:
%%capture
# @title Extract Data
import os
if os.path.exists('colab_lipid_gnn_subset.zip'):
    !unzip -o colab_lipid_gnn_subset.zip
    print('Data extraction complete.')
else:
    print('Warning: colab_lipid_gnn_subset.zip not found! Please upload it.')

In [ ]:
# @title Path & Environment Setup
import sys
import torch
import numpy as np
from pathlib import Path
import os

# Add the current directory to sys.path so we can import lipid_gnn
sys.path.append(os.getcwd())

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# @title Redefined Data Loading for Flat Structure
from tqdm.notebook import tqdm
from torch_geometric.loader import DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from colab_lipid_gnn_subset.lipid_gnn.lipid_graph import MartiniHeteroGraphBuilder
from colab_lipid_gnn_subset.lipid_gnn.functions_emil.functions import pkl_load
import numpy as np
import gc

def load_colab_data(target_properties=['lipid_packing', 'thickness'], num_frames=10):
    """
    Optimized loader for the flat 'colab_subset' structure.
    """
    graphs = []
    base_dir = Path('colab_lipid_gnn_subset')
    sims_dir = base_dir / 'sims'
    props_dir = base_dir / 'properties'
    
    # Find all compositions by searching for TPR files
    tpr_files = list(sims_dir.glob('*_prun.tpr'))
    
    for tpr_path in tqdm(tpr_files, desc="Building graphs"):
        comp = tpr_path.name.replace('_prun.tpr', '')
        xtc_path = sims_dir / f"{comp}_prun.xtc"
        prop_path = props_dir / f"{comp}.h5"
        
        if not (xtc_path.exists() and prop_path.exists()):
            continue
            
        builder = MartiniHeteroGraphBuilder(
            topology_file=str(tpr_path), 
            trajectory_file=str(xtc_path), 
            spatial_cutoff=11.0, 
            ff_params_path=base_dir / 'resources/martini_ff_params.json',
            ff_edge_params_path=base_dir / 'resources/martini_ff_edge_params.json',
            ff_node_mapping_path=base_dir / 'resources/martini_ff_node_mapping.json'
        )
        
        n_frames = builder.u.trajectory.n_frames
        # Select evenly spaced indices across the entire trajectory
        if n_frames <= num_frames:
            sampled_indices = range(n_frames)
        else:
            sampled_indices = np.linspace(0, n_frames - 1, num_frames, dtype=int)
            

        try:
            mean_dict, _ = pkl_load(prop_path, verbose=False)
            target_vec = [mean_dict[prop] for prop in target_properties]
            
            for f_idx in sampled_indices:
                hetero_data = builder.process_frame(frame_idx=int(f_idx))
                hetero_data.y = torch.tensor([target_vec], dtype=torch.float)
                graphs.append(hetero_data)
            # Clean up graph builder and free space
            del builder
            gc.collect()
        except Exception as e:
            print(f"Error loading properties for {comp}: {e}")
            
    if len(graphs) == 0:
        raise ValueError("No graphs were successfully built. Check your data paths and property files.")
    
    # Fit scaler and load graphs
    train_graphs, test_graphs = train_test_split(graphs, test_size=0.2, random_state=42)

    scaler = StandardScaler()
    train_y = torch.cat([g.y for g in train_graphs], dim=0).numpy()
    scaler.fit(train_y)
    
    for g in train_graphs:
        g.y = torch.tensor(scaler.transform(g.y.numpy()), dtype=torch.float)
    for g in test_graphs:
        g.y = torch.tensor(scaler.transform(g.y.numpy()), dtype=torch.float)

        
    return train_graphs, test_graphs

print('Loader defined.')

In [ ]:
# Setup wandb
import wandb
wandb.login()

In [ ]:
# comp_mode options: gnn_only, gnn_plus_comp, comp_only
param_config = {
    "properties": ['lipid_packing', 'thickness'],
    "hidden_dim": 32,
    "num_layers": 2,
    "epochs": 100,
    "batch_size": 2,
    "learning_rate": 5e-4,
    "weight_decay": 5e-3,
    "comp_mode": "comp_only",
    "seed": 1,
    "num_frames": 1
}

In [ ]:
# @title Training Loop Logic

import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import r2_score

from colab_lipid_gnn_subset.lipid_gnn.membrane_prop_gnn import MembranePropertyGNN
from colab_lipid_gnn_subset.lipid_gnn.plotting import plot_property_accuracies
from colab_lipid_gnn_subset.lipid_gnn.lipid_graph import LIPID_COMP_DIM

def train(param_config, train_graphs, test_graphs):
    # Reproducibility
    seed = param_config["seed"]
    torch.manual_seed(seed)
    np.random.seed(seed)

    name = f"{param_config['comp_mode']}_frames_{param_config['num_frames']}_seed_{param_config['seed']}"
    wandb.init(
        project=f"lipid_gnn_{param_config["properties"]}",
        name=name,
        config=param_config
    )

    properties = param_config["properties"]
    
    bs = param_config["batch_size"]
    train_loader = DataLoader(train_graphs, batch_size=bs, shuffle=True)
    test_loader = DataLoader(test_graphs, batch_size=bs, shuffle=False)
    
    # If gnn_only comp_dim = 0 else = LIPID_COMP_DIM
    comp_mode = param_config["comp_mode"]
    if comp_mode in ["gnn_plus_comp", "comp_only"]:
        comp_dim = LIPID_COMP_DIM
        use_comp = True
    else:
        comp_dim = 0
        use_comp = False

    model = MembranePropertyGNN(
        in_channels=4, 
        hidden_dim=param_config["hidden_dim"], 
        out_dim=len(properties), 
        num_layers=param_config["num_layers"], 
        comp_dim=comp_dim
    ).to(device)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=param_config["learning_rate"], 
                                 weight_decay=param_config["weight_decay"])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)
    criterion = torch.nn.MSELoss()

    def _get_comp_vec(batch):
        """Extract and stack per-graph composition vectors from a batch."""
        if not use_comp:
            return None
        # batch.comp_vec is shape [batch_size, LIPID_COMP_DIM] after PyG batching
        return batch.comp_vec.to(device)

    def _forward(batch):
        """Unified forward pass supporting all three comp_modes."""
        comp_vec = _get_comp_vec(batch)
        if comp_mode == 'comp_only':
            # Skip message passing; feed comp_vec directly through an identity GNN
            # by passing a zero x_dict — the comp_vec in the MLP carries all signal.
            # We still run the GNN, but its pooled output is dominated by comp_vec.
            pass  # comp_vec will be appended; GNN output acts as learned noise
        edge_attr_dict = batch.edge_attr_dict if hasattr(batch, 'edge_attr_dict') else None
        return model(batch.x_dict, batch.edge_index_dict, batch.batch_dict,
                     edge_attr_dict, comp_vec=comp_vec)

    for epoch in range(1, param_config["epochs"]+1):
        model.train()
        total_epoch_loss = 0
        individual_epoch_losses = torch.zeros(len(properties)).to(device)
        for batch_idx, batch in enumerate(train_loader):
            batch = batch.to(device)
            optimizer.zero_grad()
            out = _forward(batch)
            target = batch.y
            loss = criterion(out, target)
            loss.backward()
            optimizer.step()
            # Log total batch loss
            batch_total_loss = loss.item()
            if batch_idx % 10 == 0:
                wandb.log({"batch/loss": batch_total_loss})
            total_epoch_loss += batch_total_loss * batch.num_graphs
            # Log individual batch losses
            individual_epoch_losses += torch.mean((out - target)**2, dim=0) * batch.num_graphs

        # Compute average losses, total and for every property
        avg_train_loss = total_epoch_loss / len(train_graphs)
        avg_ind_train_loss = individual_epoch_losses.cpu().detach().numpy() / len(train_graphs)
        
        model.eval()
        total_test_loss = 0
        individual_test_losses = torch.zeros(len(properties)).to(device)
        all_preds, all_actuals = [], []
        with torch.no_grad():
            for batch in test_loader:
                batch = batch.to(device)
                predictions = _forward(batch)
                target = batch.y
                loss = criterion(predictions, target)
                total_test_loss += loss.item() * batch.num_graphs
                # Log individual test losses
                individual_test_losses += torch.mean((predictions - target)**2, dim=0) * batch.num_graphs
                # Collect prediction and target
                all_preds.append(predictions.cpu().numpy())
                all_actuals.append(target.detach().cpu().numpy())

        # Compute average test loss and update scheduler
        avg_test_loss = total_test_loss / len(test_graphs)
        scheduler.step(avg_test_loss)
        # Compute average individual test loss
        avg_ind_test_loss = individual_test_losses.cpu().detach().numpy() / len(test_graphs)
        
        # Concatenate all batch into matrices
        all_preds = np.concatenate(all_preds, axis=0)
        all_actuals = np.concatenate(all_actuals, axis=0)
        # Calculate R2 for each property
        r2_scores = r2_score(all_actuals, all_preds, multioutput="raw_values")

        if epoch % 10 == 0 or epoch == 1:
            print(f'Epoch {epoch:03d} | Train MSE: {avg_train_loss:.4f} | Test MSE: {avg_test_loss:.4f}')
        
        # Epoch level single metrics
        metrics = {
            "epoch": epoch,
            "train/avg_total_loss": avg_train_loss,
            "test/avg_total_loss": avg_test_loss,
            "learning_rate": optimizer.param_groups[0]["lr"]
        }
        # Epoch level property metrics
        for i, prop_name in enumerate(properties):
            metrics[f"train/loss_{prop_name}"] = avg_ind_train_loss[i]
            metrics[f"test/loss_{prop_name}"] = avg_ind_test_loss[i]
            metrics[f"test/r2_{prop_name}"] = r2_scores[i]

        wandb.log(metrics)

    # Final Evaluation
    final_mse = np.mean((all_preds - all_actuals)**2)
    
    # Plot
    # Accuracy
    fig = plot_property_accuracies(all_actuals, all_preds, properties, final_mse)

    wandb.log({"multi_task_accuracy": wandb.Image(fig)})

    plt.close(fig)
    wandb.finish()


graph_tuples = {}
experiments = []
for num_frames in [1]:
    param_config.update({"num_frames": num_frames})
    train_graphs, test_graphs = load_colab_data(target_properties=param_config["properties"], 
                                                num_frames=param_config["num_frames"])
    graph_tuples[f"{num_frames}"] = (train_graphs, test_graphs)
    for mode in ["gnn_only"]:
        for seed in [0, 1, 2]:
            cfg = param_config.copy()
            cfg.update({
                "comp_mode": mode,
                "seed": seed,
                "num_frames": num_frames
            })
            experiments.append(cfg)

for exp_cfg in experiments:
    train_graphs, test_graphs = graph_tuples[f"{exp_cfg['num_frames']}"]
    train(exp_cfg, train_graphs, test_graphs)